In [10]:
!pip install datasets pandas numpy scikit-learn xgboost catboost nltk tqdm

In [13]:
import pandas as pd
import re
import nltk
from datasets import load_dataset
from nltk.corpus import stopwords
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings

warnings.filterwarnings('ignore')

# STAGE 1: DATA EXTRACTION & PREPROCESSING
print("STAGE 1: Loading and Preprocessing Data...")
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
print("Initializing dataset stream...")
streamed_dataset = load_dataset("winterForestStump/10-K_sec_filings", split="001", streaming=True)
target_rows = 2500
data_list = []
print(f"Fetching and cleaning {target_rows} filings on the fly...")
for i, example in enumerate(tqdm(streamed_dataset, total=target_rows)):
    if i >= target_rows:
        break
    # Combine 'text', 'Business', and 'Risk Factors' to ensure content if 'text' is empty
    raw_text = str(example.get('text', '') or \
                   example.get('Business', '') or \
                   example.get('Risk Factors', '') or \
                   example.get('Unresolved Staff Comments', '') or \
                   example.get('Properties', '') or \
                   example.get('Legal Proceedings', '') or \
                   example.get('Mine Safety Disclosures', '') or \
                   example.get('Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer P', ''))[:15000]
    text = re.sub(r'<[^>]+>', ' ', raw_text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    words = text.split()
    cleaned_text = " ".join([w for w in words if w not in stop_words and len(w) > 2])
    data_list.append({
        'cleaned_text': cleaned_text
    })
df = pd.DataFrame(data_list)
print(f" Successfully loaded and cleaned {len(df)} rows!\n")

# STAGE 2
print("STAGE 2: (TF-IDF)...")
tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X_features = tfidf.fit_transform(df['cleaned_text'])
print(f" Feature matrix shape: {X_features.shape}")

STAGE 1: Loading and Preprocessing Data...
Initializing dataset stream...
Fetching and cleaning 2500 filings on the fly...


100%|██████████| 2500/2500 [00:13<00:00, 184.08it/s]


 Successfully loaded and cleaned 2500 rows!

STAGE 2: (TF-IDF)...
 Feature matrix shape: (2500, 5000)


In [14]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
# STAGE 3:
print("STAGE 3: Generating Risk/Sentiment labels using VADER...")
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def assign_risk_label(text):
    score = sia.polarity_scores(text)['compound']
    if score <= -0.05:
        return 0  # Class 0: High Risk / Negative
    elif score >= 0.05:
        return 2  # Class 2: Low Risk / Positive
    else:
        return 1  # Class 1: Medium Risk / Neutral

df['Target'] = df['cleaned_text'].apply(assign_risk_label)
y_labels = df['Target']
print("\nLabel Distribution:")
print(df['Target'].value_counts().rename(index={0: 'High Risk (0)', 1: 'Medium Risk (1)', 2: 'Low Risk (2)'}))


# STAGE 4:
print("\nSTAGE 4: Training and Comparing Models...")
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)
target_names = ['High Risk (0)', 'Medium Risk (1)', 'Low Risk (2)']

print("\nTraining AdaBoost Classifier...")
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_model.fit(X_train, y_train)
ada_preds = ada_model.predict(X_test)
print(" AdaBoost Accuracy:", f"{accuracy_score(y_test, ada_preds) * 100:.2f}%")
print("AdaBoost Confusion Matrix:\n", confusion_matrix(y_test, ada_preds))

print("\nTraining XGBoost Classifier...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
print(" XGBoost Accuracy:", f"{accuracy_score(y_test, xgb_preds) * 100:.2f}%")
print("XGBoost Confusion Matrix:\n", confusion_matrix(y_test, xgb_preds))

print("\nTraining CatBoost Classifier...")
cat_model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
cat_model.fit(X_train, y_train)
cat_preds = cat_model.predict(X_test)
print(" CatBoost Accuracy:", f"{accuracy_score(y_test, cat_preds) * 100:.2f}%")
print("CatBoost Confusion Matrix:\n", confusion_matrix(y_test, cat_preds))

# FINAL
print(" DETAILED CLASSIFICATION REPORTS (Precision, Recall, F1) ")

print("\n--- ADABOOST METRICS ---")
print(classification_report(y_test, ada_preds, target_names=target_names, zero_division=0))

print("\n--- XGBOOST METRICS ---")
print(classification_report(y_test, xgb_preds, target_names=target_names, zero_division=0))

print("\n--- CATBOOST METRICS ---")
print(classification_report(y_test, cat_preds, target_names=target_names, zero_division=0))

STAGE 3: Generating Risk/Sentiment labels using VADER...

Label Distribution:
Target
Low Risk (2)       1721
Medium Risk (1)     718
High Risk (0)        61
Name: count, dtype: int64
------------------------------------------------------------

STAGE 4: Training and Comparing Models...

Training AdaBoost Classifier...
 AdaBoost Accuracy: 83.40%
AdaBoost Confusion Matrix:
 [[  2   0   8]
 [  0 143   8]
 [ 54  13 272]]

Training XGBoost Classifier...
 XGBoost Accuracy: 95.20%
XGBoost Confusion Matrix:
 [[  0   0  10]
 [  0 146   5]
 [  1   8 330]]

Training CatBoost Classifier...
 CatBoost Accuracy: 95.80%
CatBoost Confusion Matrix:
 [[  3   1   6]
 [  0 146   5]
 [  0   9 330]]

 DETAILED CLASSIFICATION REPORTS (Precision, Recall, F1) 

--- ADABOOST METRICS ---
                 precision    recall  f1-score   support

  High Risk (0)       0.04      0.20      0.06        10
Medium Risk (1)       0.92      0.95      0.93       151
   Low Risk (2)       0.94      0.80      0.87       339


In [16]:
import joblib

print("Saving models for deployment...")
joblib.dump(xgb_model, 'best_model.joblib')
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
print("Saved successfully!")

Saving models for deployment...
Saved successfully!
